# Climate 2014-2024 (December): Sea Level, CO2, CH4, and Arctic Ice Analysis

Uses sea_level_co2.csv containing monthly climate data:

1. Filter December data for each year from 2014 through 2024 for sea level, CO2 concentration, CH4 concentration, and Arctic ice extent.
2. Calculate lower_values as year-over-year differences in December sea level (mm), with 2014 baseline set to 0.
3. Normalize three series attributes to [0, 1] range using global min/max from the selected December data:
   - co2_ppm: CO2 parts per million
   - ch4_ppb: CH4 parts per billion
   - arctic_ice_extent_mkm2: Arctic ice extent in million km2
4. Build axis_order centered at 0 with symmetric expansion for balanced display of sea level changes.

Source: https://www.kaggle.com/datasets/sergionefedov/climate-and-weather-anomalies-196-cities-75-years?resource=download&select=temperature_anomalies.csv

In [35]:
import pandas as pd
df = pd.read_csv('sea_level_co2.csv')

years = list(range(2014, 2025))

# Filter December data for 2014-2024
df_dec = df[(df['year'].isin(years)) & (df['month'] == 12)].copy()
df_dec = df_dec.sort_values(['year']).reset_index(drop=True)

missing_years = sorted(set(years) - set(df_dec['year'].tolist()))

print(f"Selected rows (Dec 2014-2024): {len(df_dec)}")
if missing_years:
    print(f"Missing December data for years: {missing_years}")

print(df_dec[['date', 'year', 'month', 'co2_ppm', 'ch4_ppb', 'arctic_ice_extent_mkm2', 'sea_level_mm']])

Selected rows (Dec 2014-2024): 11
          date  year  month  co2_ppm  ch4_ppb  arctic_ice_extent_mkm2  \
0   2014-12-01  2014     12   425.11   1051.5                    9.58   
1   2015-12-01  2015     12   427.20   1061.1                    9.58   
2   2016-12-01  2016     12   429.26   1069.7                    9.38   
3   2017-12-01  2017     12   431.12   1068.5                    8.97   
4   2018-12-01  2018     12   433.35   1079.1                    9.31   
5   2019-12-01  2019     12   435.22   1084.4                    9.33   
6   2020-12-01  2020     12   437.47   1086.9                    9.35   
7   2021-12-01  2021     12   439.44   1096.7                    8.75   
8   2022-12-01  2022     12   441.79   1101.4                    9.13   
9   2023-12-01  2023     12   443.82   1106.5                    8.55   
10  2024-12-01  2024     12   446.06   1116.6                    8.70   

    sea_level_mm  
0          47.82  
1          48.31  
2          50.65  
3          53

## 1. Data filtering and year-over-year differences (December data)

In [36]:
import json
import math
from pathlib import Path

# Extract required data from December rows (2014-2024)
times = df_dec['year'].astype(int).tolist()  # 2014-2024
series_cols = ['co2_ppm', 'ch4_ppb', 'arctic_ice_extent_mkm2']

# Calculate lower_values as year-over-year difference in December sea level
# Baseline: first year = 0
lower_values = []
prev_value = None

for idx, row in df_dec.iterrows():
    current_value = float(row['sea_level_mm'])
    if idx == 0:
        lower_values.append(0.0)
    else:
        diff = current_value - prev_value
        lower_values.append(round(diff, 2))
    
    prev_value = current_value

# Normalize series data to [0, 1] range
def normalize_01(values):
    """Normalize values to [0, 1] range."""
    col_data = values.astype(float)
    min_val = col_data.min()
    max_val = col_data.max()
    
    if max_val == min_val:
        return [0.0] * len(values)
    
    normalized = [(v - min_val) / (max_val - min_val) for v in col_data]
    return normalized

# Normalize series data
series_data = []
for col in series_cols:
    normalized_values = normalize_01(df_dec[col])
    series_data.append({
        'id': col,
        'times': times,
        'values': [round(v, 6) for v in normalized_values]
    })

print("Series data normalized:")
for s in series_data:
    print(f"{s['id']}: {s['values']}")

print(f"\nLower values (year-over-year Dec difference, first year=0): {lower_values}")


Series data normalized:
co2_ppm: [np.float64(0.0), np.float64(0.099761), np.float64(0.198091), np.float64(0.286874), np.float64(0.393317), np.float64(0.482578), np.float64(0.589976), np.float64(0.68401), np.float64(0.796181), np.float64(0.893079), np.float64(1.0)]
ch4_ppb: [np.float64(0.0), np.float64(0.147465), np.float64(0.27957), np.float64(0.261137), np.float64(0.423963), np.float64(0.505376), np.float64(0.543779), np.float64(0.694316), np.float64(0.766513), np.float64(0.844854), np.float64(1.0)]
arctic_ice_extent_mkm2: [np.float64(1.0), np.float64(1.0), np.float64(0.805825), np.float64(0.407767), np.float64(0.737864), np.float64(0.757282), np.float64(0.776699), np.float64(0.194175), np.float64(0.563107), np.float64(0.0), np.float64(0.145631)]

Lower values (year-over-year Dec difference, first year=0): [0.0, 0.49, 2.34, 3.14, 5.14, 4.55, 0.51, 6.59, 0.53, 8.16, -2.56]


## 2. Normalize series to [0, 1] range and calculate year-over-year lower_values

In [37]:
def build_axis_order(values, step=2):
    """
    Build axis_order with following requirements:
    - Center value must be 0
    - Left side: negative numbers
    - Right side: positive numbers
    - Symmetric expansion for balanced display
    """
    min_val = min(values)
    max_val = max(values)
    
    # Find maximum absolute value range
    abs_max = max(abs(min_val), abs(max_val))
    
    # Build symmetrically based on absolute maximum
    abs_max_rounded = step * math.ceil(abs_max / step)
    
    # Generate negative part (from -abs_max_rounded to -step)
    negative_values = list(range(-abs_max_rounded, 0, step))
    
    # Generate positive part (from step to abs_max_rounded)
    positive_values = list(range(step, abs_max_rounded + 1, step))
    
    # Combine: negative + 0 + positive
    axis_order = negative_values + [0] + positive_values
    return axis_order

axis_order = build_axis_order(lower_values, step=2)
print(f"Axis order length: {len(axis_order)}")
print(f"Axis order: {axis_order}")
print(f"Center value confirmed 0: {axis_order[len(axis_order)//2] == 0}")
print(f"Min value: {axis_order[0]}, Max value: {axis_order[-1]}")


Axis order length: 11
Axis order: [-10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10]
Center value confirmed 0: True
Min value: -10, Max value: 10


## 3. Build axis order centered at 0

In [38]:
# Build final JSON
glyph_json = {
    "paths": [
        {
            "label": "Climate 2014-2024",
            "nodes": times,
            "times": times,
            "lower_values": lower_values,
        }
    ],
    "axis_order": axis_order,
    "series_by_path": [
        {
            "path_index": 0,
            "series": series_data
        }
    ],
}

# Save to file
_repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path('/Users/siyuanzhao/Documents/GitHub/Loom_General')
]
repo_root = next((p for p in _repo_candidates if (p / "Frontend" / "src" / "data").exists()), Path.cwd())
out_path = repo_root / "Frontend" / "src" / "data" / "climateGlyph.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(glyph_json, f, indent=2)

print("Output path:", out_path.resolve())
print("\nJSON content:")
print(json.dumps(glyph_json, indent=2))


Output path: /Users/siyuanzhao/Documents/GitHub/Loom_General/Frontend/src/data/climateGlyph.json

JSON content:
{
  "paths": [
    {
      "label": "Climate 2014-2024",
      "nodes": [
        2014,
        2015,
        2016,
        2017,
        2018,
        2019,
        2020,
        2021,
        2022,
        2023,
        2024
      ],
      "times": [
        2014,
        2015,
        2016,
        2017,
        2018,
        2019,
        2020,
        2021,
        2022,
        2023,
        2024
      ],
      "lower_values": [
        0.0,
        0.49,
        2.34,
        3.14,
        5.14,
        4.55,
        0.51,
        6.59,
        0.53,
        8.16,
        -2.56
      ]
    }
  ],
  "axis_order": [
    -10,
    -8,
    -6,
    -4,
    -2,
    0,
    2,
    4,
    6,
    8,
    10
  ],
  "series_by_path": [
    {
      "path_index": 0,
      "series": [
        {
          "id": "co2_ppm",
          "times": [
            2014,
            2015,
        